# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [15]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import gradio as gr
from IPython.display import Markdown, display, update_display


In [16]:
from google import genai
from google.genai import types

In [17]:
from openai import OpenAI

In [18]:
from dotenv import load_dotenv
load_dotenv()

CONFIG = {
    "company_name": "HuggingFace",
    "url": "https://huggingface.co",
    "api_key": os.getenv("GEMINI_API_KEY"),
    "model": "gemini-2.0-flash"
}

In [19]:

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        try:
            self.url = url
            response = requests.get(url, headers=headers)
            self.body = response.content
            soup = BeautifulSoup(self.body, 'html.parser')
            self.title = soup.title.string if soup.title else "No title found"
            if soup.body:
                for irrelevant in soup.body(["script", "style", "img", "input"]):
                    irrelevant.decompose()
                self.text = soup.body.get_text(separator="\n", strip=True)
            else:
                self.text = ""
            links = [link.get('href') for link in soup.find_all('a')]
            self.links = [link for link in links if link]
        except:
            print("website error")
    def get_contents(self):
        try:
            return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"
        except:
            return ""

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [20]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [21]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [22]:
def get_links(company_name, url):
    gemini_via_openai = OpenAI(
        api_key=CONFIG["api_key"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    website = Website(url)
    prompt = f"{link_system_prompt}\n\n{get_links_user_prompt(website)}"
    messages = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(website)}
    ]
    response = gemini_via_openai.chat.completions.create(
        messages=messages,
        model=CONFIG["model"]
    )
    try:
        return json.loads(response.choices[0].message['content'])
    except json.JSONDecodeError:
        print("Warning: Invalid JSON response:", response.choices[0].message['content'])
        return {"links": []}
    except Exception as e:
        print(f"Error generating content: {e}")
        return {"links": []}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [23]:
def get_all_details(company_name, url):
    """Fetch the landing page and relevant links' content."""
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(company_name, url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n" 
        result += Website(link["url"]).get_contents()
    return result


In [24]:
system_prompt = """You are an assistant that analyzes the contents of several relevant pages from a company website 
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond ONLY in markdown format. Do not show your reasoning or analysis steps.
Include only the formatted brochure with details about company culture, customers and careers/jobs if available."""

In [25]:
def get_brochure_user_prompt(company_name,url):
    """Generate the user prompt for creating the brochure."""
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown. Mention any relevant links.\n"
    user_prompt += get_all_details(company_name,url)
    user_prompt = user_prompt[:5_000] 
    return user_prompt

In [26]:
def create_brochure(company_name, url):
    gemini_via_openai = OpenAI(
        api_key=CONFIG["api_key"], 
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]
    response = gemini_via_openai.chat.completions.create(
        model=CONFIG["model"],
        messages=messages,
        # stream=True
    )
    try:
        result = response.choices[0].message.content
        result = result.replace("```", "").replace("markdown", "")
        return result
    except Exception as e:
        print(f"Error generating content: {e}")
        return "Error generating brochure."

In [27]:
def launch_interface():
    view = gr.Interface(
        fn=create_brochure,
        inputs=[
            gr.Textbox(label="Gemini Brochure name:"),
            gr.Textbox(label="Landing page URL including http:// or https://"),
        ],
        outputs=[gr.Markdown(label="Brochure:")],
        title="Company Brochure Generator",
        description="Enter a company name and URL to generate a professional brochure.",
        flagging_mode="never"
    )
    view.launch()

In [28]:
if __name__ == "__main__":
    launch_interface()

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


Error generating content: 'ChatCompletionMessage' object is not subscriptable
Found links: {'links': []}


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>